In [1]:
from utils_sql import execute_query, create_connection_to_vector_db
from my_fin_rep_utils import get_fin_reports_async, load_api_tokens, create_insert_table_db_async
from datetime import datetime, timedelta
import asyncio
import pandas as pd

In [ ]:
4333899257,
4334946766,
4333631745,
4335165642,
4335639007,
4334890329,
4334177405,
4335561608,
4335172251,
4334946076,
4335831149,
4335765212,
4335994383,
4335217001,
4334366446,
4335732659,
4334709158,
4334668615,
4333869075,
4333665608,
4334614039,
4335841044,
4334421181,
4335633248,
4333887368,
4336211786,
4335220967,
4335614863,
4333985462,
4335682112,
4335530530,
4335584865,
4334524370,
4334589248,
4336159028,
4335552338,
4334159854,
4335760742,
4334394443,
4334965929,
4334614036,
4334857615,
4335984158,
4334424356,
4333647380,
4335793392,
4334751256,
4334222141,
4334203334,
4335380608,
4335614865,
4335782619,
4335615800,
4333899243,
4334087190,
4333883661,
4335172250,
4336215956

In [5]:
connection = create_connection_to_vector_db()
query = """SELECT *
    FROM card_data"""
execute_query(connection, query)

Соединение с БД PostgreSQL успешно установлено в 2025-11-07-14:M
Запрос успешно выполнен в 2025-11-07


In [10]:
# Это нужно для соблюдения рейт-лимита Wildberries API
next_request_time = {}
# request_lock используется для синхронизации доступа к next_request_time
# Это нужно, чтобы избежать гонок при обновлении времени следующего запроса
# Если несколько корутин одновременно попытаются обновить время, это может привести к ошибкам
request_lock = asyncio.Lock()
# semaphore ограничивает количество одновременных запросов к API
# semaphore = asyncio.Semaphore(8)  # максимум 8 одновременных запросов

num_weeks = 2
today = datetime.today()
weekday = today.weekday()
current_sunday = today - timedelta(days=(weekday + 1) % 7)
processed_weeks = 0
for week in range(num_weeks-1, num_weeks):
        current_monday = current_sunday - timedelta(days=6)
        # date_from = current_monday.strftime('%Y-%m-%d')
        # date_to = current_sunday.strftime('%Y-%m-%d')
        date_from = '2025-11-03'
        date_to = '2025-11-09'

        # logging.info(f"📅 Загружаем неделю {week + 1}/{num_weeks}: {date_from} – {date_to}")

        tasks = [
            get_fin_reports_async(account, token, date_from, date_to, request_lock, next_request_time)
            for account, token in load_api_tokens().items()
        ]

        weekly_results = await asyncio.gather(*tasks, return_exceptions=True)

        table_name = 'fin_reports_full'
        columns_type = {
            "realizationreport_id": "INTEGER",
            "date_from": "DATE",
            "date_to": "DATE",
            "create_dt": "DATE",
            "currency_name": "TEXT",
            "suppliercontract_code": "TEXT",
            "rrd_id": "BIGINT",
            "gi_id": "BIGINT",
            "dlv_prc": "TEXT",
            "fix_tariff_date_from": "DATE",
            "fix_tariff_date_to": "DATE",
            "subject_name": "TEXT",
            "nm_id": "BIGINT",
            "brand_name": "TEXT",
            "sa_name": "TEXT",
            "ts_name": "TEXT",
            "barcode": "TEXT",
            "doc_type_name": "TEXT",
            "quantity": "INTEGER",
            "retail_price": "NUMERIC(12,2)",
            "retail_amount": "NUMERIC(12,2)",
            "sale_percent": "SMALLINT",
            "commission_percent": "NUMERIC(5,2)",
            "office_name": "TEXT",
            "supplier_oper_name": "TEXT",
            "order_dt": "TIMESTAMP",
            "sale_dt": "TIMESTAMP",
            "rr_dt": "DATE",
            "shk_id": "BIGINT",
            "retail_price_withdisc_rub": "NUMERIC(12,2)",
            "delivery_amount": "INTEGER",
            "return_amount": "INTEGER",
            "delivery_rub": "NUMERIC(12,2)",
            "gi_box_type_name": "TEXT",
            "product_discount_for_report": "NUMERIC(5,2)",
            "supplier_promo": "NUMERIC(12,2)",
            "ppvz_spp_prc": "NUMERIC(5,2)",
            "ppvz_kvw_prc_base": "NUMERIC(5,2)",
            "ppvz_kvw_prc": "NUMERIC(5,2)",
            "sup_rating_prc_up": "NUMERIC(5,2)",
            "is_kgvp_v2": "BOOLEAN",
            "ppvz_sales_commission": "NUMERIC(12,2)",
            "ppvz_for_pay": "NUMERIC(12,2)",
            "ppvz_reward": "NUMERIC(12,2)",
            "acquiring_fee": "NUMERIC(12,2)",
            "acquiring_percent": "NUMERIC(5,2)",
            "payment_processing": "TEXT",
            "acquiring_bank": "TEXT",
            "ppvz_vw": "NUMERIC(12,2)",
            "ppvz_vw_nds": "NUMERIC(12,2)",
            "ppvz_office_name": "TEXT",
            "ppvz_office_id": "INTEGER",
            "ppvz_supplier_id": "INTEGER",
            "ppvz_supplier_name": "TEXT",
            "ppvz_inn": "TEXT",
            "declaration_number": "TEXT",
            "bonus_type_name": "TEXT",
            "sticker_id": "TEXT",
            "site_country": "TEXT",
            "srv_dbs": "BOOLEAN",
            "penalty": "NUMERIC(12,2)",
            "additional_payment": "NUMERIC(12,2)",
            "rebill_logistic_cost": "NUMERIC(12,2)",
            "rebill_logistic_org": "TEXT",
            "storage_fee": "NUMERIC(12,2)",
            "deduction": "NUMERIC(12,2)",
            "acceptance": "NUMERIC(12,2)",
            "assembly_id": "BIGINT",
            "srid": "TEXT",
            "report_type": "SMALLINT",
            "is_legal_entity": "BOOLEAN",
            "trbx_id": "TEXT",
            "installment_cofinancing_amount": "NUMERIC(12,2)",
            "wibes_wb_discount_percent": "SMALLINT",
            "cashback_amount": "NUMERIC(12,2)",
            "cashback_discount": "NUMERIC(12,2)",
            "account": "VARCHAR(50)",
            }
        key_columns = ['realizationreport_id', 'rrd_id', 'srid']

        save_tasks = []
        for result in weekly_results:
            if isinstance(result, Exception):
                # logging.error(f"Ошибка при загрузке недели {week + 1}: {result}")
                continue
            if isinstance(result, pd.DataFrame) and not result.empty:
                save_tasks.append(
                    create_insert_table_db_async(result, table_name, columns_type, key_columns)
                )

        if save_tasks:
            await asyncio.gather(*save_tasks, return_exceptions=True)
        processed_weeks += 1
        # logging.info(f"✅ Завершена обработка недели {week + 1}/{num_weeks}")

        current_sunday -= timedelta(days=7)

🔍 Оганесян | Запрос за 2025-11-03 – 2025-11-09
Получены сырые данные (50000 строк): [{'realizationreport_id': 531376981, 'date_from': '2025-11-03', 'date_to': '2025-11-09', 'create_dt': '2025-11-10', 'currency_name': 'RUB', 'suppliercontract_code': None, 'rrd_id': 3048816968301, 'gi_id': 0, 'dlv_prc': 0, 'fix_tariff_date_from': '', 'fix_tariff_date_to': '', 'subject_name': 'Овощерезки ручные', 'nm_id': 162374183, 'brand_name': 'VEGGIE SLICER', 'sa_name': 'wild399d', 'ts_name': '0', 'barcode': '2037877244365', 'doc_type_name': 'Продажа', 'quantity': 1, 'retail_price': 687, 'retail_amount': 510, 'sale_percent': 0, 'commission_percent': 31, 'office_name': 'Центральный федеральный округ МП', 'supplier_oper_name': 'Продажа', 'order_dt': '2025-10-15T14:08:26Z', 'sale_dt': '2025-11-03T00:07:53Z', 'rr_dt': '2025-11-03', 'shk_id': 43103062547, 'retail_price_withdisc_rub': 687, 'delivery_amount': 0, 'return_amount': 0, 'delivery_rub': 0, 'gi_box_type_name': '', 'product_discount_for_report': 0, 

Создан DataFrame с колонками: ['realizationreport_id', 'date_from', 'date_to', 'create_dt', 'currency_name', 'suppliercontract_code', 'rrd_id', 'gi_id', 'dlv_prc', 'fix_tariff_date_from', 'fix_tariff_date_to', 'subject_name', 'nm_id', 'brand_name', 'sa_name', 'ts_name', 'barcode', 'doc_type_name', 'quantity', 'retail_price', 'retail_amount', 'sale_percent', 'commission_percent', 'office_name', 'supplier_oper_name', 'order_dt', 'sale_dt', 'rr_dt', 'shk_id', 'retail_price_withdisc_rub', 'delivery_amount', 'return_amount', 'delivery_rub', 'gi_box_type_name', 'product_discount_for_report', 'supplier_promo', 'ppvz_spp_prc', 'ppvz_kvw_prc_base', 'ppvz_kvw_prc', 'sup_rating_prc_up', 'is_kgvp_v2', 'ppvz_sales_commission', 'ppvz_for_pay', 'ppvz_reward', 'acquiring_fee', 'acquiring_percent', 'payment_processing', 'acquiring_bank', 'ppvz_vw', 'ppvz_vw_nds', 'ppvz_office_name', 'ppvz_office_id', 'ppvz_supplier_id', 'ppvz_supplier_name', 'ppvz_inn', 'declaration_number', 'sticker_id', 'site_country

Создан DataFrame с колонками: ['realizationreport_id', 'date_from', 'date_to', 'create_dt', 'currency_name', 'suppliercontract_code', 'rrd_id', 'gi_id', 'dlv_prc', 'fix_tariff_date_from', 'fix_tariff_date_to', 'subject_name', 'nm_id', 'brand_name', 'sa_name', 'ts_name', 'barcode', 'doc_type_name', 'quantity', 'retail_price', 'retail_amount', 'sale_percent', 'commission_percent', 'office_name', 'supplier_oper_name', 'order_dt', 'sale_dt', 'rr_dt', 'shk_id', 'retail_price_withdisc_rub', 'delivery_amount', 'return_amount', 'delivery_rub', 'gi_box_type_name', 'product_discount_for_report', 'supplier_promo', 'ppvz_spp_prc', 'ppvz_kvw_prc_base', 'ppvz_kvw_prc', 'sup_rating_prc_up', 'is_kgvp_v2', 'ppvz_sales_commission', 'ppvz_for_pay', 'ppvz_reward', 'acquiring_fee', 'acquiring_percent', 'payment_processing', 'acquiring_bank', 'ppvz_vw', 'ppvz_vw_nds', 'ppvz_office_name', 'ppvz_office_id', 'ppvz_supplier_id', 'ppvz_supplier_name', 'ppvz_inn', 'declaration_number', 'sticker_id', 'site_country

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83634 entries, 0 to 83633
Data columns (total 80 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   realizationreport_id            83634 non-null  object
 1   date_from                       83634 non-null  object
 2   date_to                         83634 non-null  object
 3   create_dt                       83634 non-null  object
 4   currency_name                   83634 non-null  object
 5   suppliercontract_code           0 non-null      object
 6   rrd_id                          83634 non-null  object
 7   gi_id                           83634 non-null  object
 8   dlv_prc                         83634 non-null  object
 9   fix_tariff_date_from            0 non-null      object
 10  fix_tariff_date_to              0 non-null      object
 11  subject_name                    83634 non-null  object
 12  nm_id                           83634 non-null

In [9]:
result[result['cashback_discount'] == result['cashback_discount'].max()]

,realizationreport_id,date_from,date_to,create_dt,currency_name,suppliercontract_code,rrd_id,gi_id,dlv_prc,fix_tariff_date_from,...,installment_cofinancing_amount,wibes_wb_discount_percent,cashback_amount,cashback_discount,cashback_commission_change,order_uid,payment_schedule,bonus_type_name,rebill_logistic_org,account
12075,531376981,2025-11-03,2025-11-09,2025-11-10,RUB,None,3048917044047,0,0,None,...,0,0,0,1111.0,0,,0,Компенсация скидки по программе лояльности,None,Оганесян


In [5]:
def find_problematic_values(df):
    """
    Находит значения, которые не влезут в NUMERIC(5,2)
    """
    # NUMERIC(5,2) ограничение: максимум 999.99
    max_allowed = 999.99
    
    problematic_fields = {}
    
    # Проверяем все числовые поля которые могут быть деньгами или процентами
    money_percent_columns = [
        'retail_price', 'retail_amount', 'sale_percent', 'commission_percent',
        'retail_price_withdisc_rub', 'delivery_amount', 'return_amount', 
        'delivery_rub', 'product_discount_for_report', 'supplier_promo',
        'ppvz_spp_prc', 'ppvz_kvw_prc_base', 'ppvz_kvw_prc', 'sup_rating_prc_up',
        'ppvz_sales_commission', 'ppvz_for_pay', 'ppvz_reward', 'acquiring_fee',
        'payment_processing', 'ppvz_vw', 'ppvz_vw_nds', 'penalty', 'additional_payment',
        'rebill_logistic_cost', 'storage_fee', 'deduction', 'acceptance',
        'installment_cofinancing_amount', 'wibes_wb_discount_percent', 
        'cashback_amount', 'cashback_discount', 'cashback_commission_change'
    ]
    
    for col in money_percent_columns:
        if col in df.columns:
            # Временно преобразуем в числа
            temp_series = pd.to_numeric(
                df[col].astype(str).str.replace(',', '.'), 
                errors='coerce'
            )
            
            # Ищем значения больше максимально допустимого
            too_large = temp_series[temp_series > max_allowed]
            if len(too_large) > 0:
                problematic_fields[col] = {
                    'count': len(too_large),
                    'max_value': temp_series.max(),
                    'problematic_values': too_large.head().tolist()
                }
                print(f"⚠️  {col}: {len(too_large)} значений > {max_allowed}, макс: {temp_series.max()}")
    
    return problematic_fields

# Использование:
problematic = find_problematic_values(result)

⚠️  retail_price: 2562 значений > 999.99, макс: 11407.0
⚠️  retail_amount: 1639 значений > 999.99, макс: 7902.0
⚠️  retail_price_withdisc_rub: 2562 значений > 999.99, макс: 11407.0
⚠️  delivery_rub: 1 значений > 999.99, макс: 1184.93
⚠️  ppvz_sales_commission: 2 значений > 999.99, макс: 1133.73
⚠️  ppvz_for_pay: 1954 значений > 999.99, макс: 9089.52
⚠️  ppvz_vw: 2 значений > 999.99, макс: 1133.725
⚠️  rebill_logistic_cost: 3 значений > 999.99, макс: 2102.63
⚠️  deduction: 18 значений > 999.99, макс: 1228531.0
⚠️  cashback_discount: 1 значений > 999.99, макс: 1111.0


In [8]:
type(weekly_results)

list